In [5]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [14]:
df = pd.read_csv(r"C:\Users\Nourhan Yehia\Desktop\Jupyter\poem generator\all_poems.csv")

df.columns

Index(['poem_id', 'poem_link', 'poem_style', 'poem_text', 'poem_title',
       'poet_cat', 'poet_id', 'poet_link', 'poet_name'],
      dtype='object')

In [15]:
df = df[df["poet_name"] == "المتنبي"]

In [17]:
df = df[df["poem_style"] == "فصحى"]

In [18]:
df = df[["poem_text"]]

In [19]:
df = df.dropna()
df = df.drop_duplicates()

In [20]:
df.shape
df.head()

,poem_text
4054,عذل العواذل حول قلبي التاءه وهوي الاحبة منه ف...
4055,القلب اعلم يا عذول بداءه واحق منك بجفنه وبماء...
4056,اتنكر يا ابن اسحق اخاءي وتحسب ماء غيري من انا...
4057,امن ازديارك في الدجي الرقباء اذ حيث كنت من ال...
4058,ماذا يقول الذي يغني يا خير من تحت ذي السماء ش...


In [21]:
def clean_text(text):
    
    text = re.sub(r'[ًٌٍَُِّْـ]', '', text)
    
    text = re.sub(r'[0-9]', '', text)
    
    text = re.sub(r'[^\u0600-\u06FF\s]', '', text)
    
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

In [22]:
df["clean_text"] = df["poem_text"].apply(clean_text)

In [26]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("aubmindlab/aragpt2-base")

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

C:\Users\Nourhan Yehia\.conda\envs\cnn_xray\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Nourhan Yehia\.cache\huggingface\hub\models--aubmindlab--aragpt2-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [27]:
sample = df["clean_text"].iloc[0]

tokens = tokenizer(sample)

print(tokens["input_ids"][:20])

[276, 5472, 4277, 57447, 1027, 10980, 325, 33826, 667, 530, 1167, 491, 1510, 305, 15466, 280, 32947, 293, 552, 1046]


In [28]:
train_texts, temp_texts = train_test_split(df["clean_text"], test_size=0.2, random_state=42)
val_texts, test_texts = train_test_split(temp_texts, test_size=0.5, random_state=42)

In [35]:
tokenizer.pad_token = tokenizer.eos_token

In [39]:
max_length = 128
stride = 64

def tokenize_with_overflow(texts, tokenizer, max_length=128, stride=64):
    input_ids_list = []
    attention_mask_list = []

    for text in texts:
        encodings = tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            stride=stride,
            return_overflowing_tokens=True,
            return_attention_mask=True
        )

        input_ids_list.extend(encodings["input_ids"])
        attention_mask_list.extend(encodings["attention_mask"])

    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_mask_list
    }

In [40]:
train_encodings = tokenize_with_overflow(train_texts, tokenizer, max_length=128, stride=64)
val_encodings = tokenize_with_overflow(val_texts, tokenizer, max_length=128, stride=64)
test_encodings = tokenize_with_overflow(test_texts, tokenizer, max_length=128, stride=64)

In [41]:
print("Train sequences:", len(train_encodings["input_ids"]))
print("Validation sequences:", len(val_encodings["input_ids"]))
print("Test sequences:", len(test_encodings["input_ids"]))

Train sequences: 918
Validation sequences: 107
Test sequences: 91


In [42]:
class PoetryDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):
        item = {
            "input_ids": torch.tensor(self.encodings["input_ids"][idx]),
            "attention_mask": torch.tensor(self.encodings["attention_mask"][idx]),
            "labels": torch.tensor(self.encodings["input_ids"][idx])
        }
        return item

    def __len__(self):
        return len(self.encodings["input_ids"])

In [44]:
train_dataset = PoetryDataset(train_encodings)
val_dataset = PoetryDataset(val_encodings)
test_dataset = PoetryDataset(test_encodings)

In [45]:
model = AutoModelForCausalLM.from_pretrained("aubmindlab/aragpt2-base")

model.safetensors:   0%|          | 0.00/553M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: aubmindlab/aragpt2-base
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [47]:
from transformers import TrainingArguments, Trainer

In [50]:
training_args = TrainingArguments(
    output_dir="./aragpt_poetry_model",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    logging_steps=50
)

In [51]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [52]:
trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
50,7.506782
100,6.636287
150,6.712946
200,6.085291
250,6.152302
300,6.436143
350,6.492438
400,6.138798
450,5.989370
500,5.875298


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1377, training_loss=5.895968634578398, metrics={'train_runtime': 952.8193, 'train_samples_per_second': 2.89, 'train_steps_per_second': 1.445, 'total_flos': 179899564032000.0, 'train_loss': 5.895968634578398, 'epoch': 3.0})

In [70]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

def generate_poem(prompt):
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    outputs = model.generate(
    **inputs,
    max_length=80,
    do_sample=True,
    top_k=40,
    top_p=0.9,
    temperature=0.7,
    repetition_penalty=1.3,
    no_repeat_ngram_size=3,
    pad_token_id=tokenizer.eos_token_id
)

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    words = text.split()

    lines = [" ".join(words[i:i+6]) for i in range(0, len(words), 6)]

    poem = "\n".join(lines)

    print(poem)

In [71]:
generate_poem("يا قلب")

يا قلب عاشق غدارا لم يكن
له الا قلوب حاءرا وان كان
لا يري نفسه في الناس ما
يراه غير مقبلا فانما يذم اذا
غضبوا فضلوا فانه لناظلوا ان يكونوا
عبيدا واذا طلبوا فلا يبغضون ولا
يقدعوا وان يكدوا فقد علموا انهم
قد خسروا فانفسهم عبيدا او خابوا
وقالوا لهم ان الموت عندهم كالاخطاء
وقد
